In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("data_with_elo.csv", index_col=0)

In [3]:
df["gameDateTimeEst"] = pd.to_datetime(df["gameDateTimeEst"])

In [4]:
df["win"] = df["win"].astype(bool)

In [5]:
df["game_date"] = df["gameDateTimeEst"].dt.date
df["last_game_played"] = df.groupby("teamName")["game_date"].shift(1)

In [6]:
df['game_date'] = pd.to_datetime(df['game_date'])
df['last_game_played'] = pd.to_datetime(df['last_game_played'])

In [7]:
df["days_rest"] = (df["game_date"] - df["last_game_played"]).dt.days
df["is_B2B"] = df["days_rest"] == 1

In [8]:
df = df.sort_values(by=["gameDateTimeEst", "gameId"])

In [9]:
rolling_features = ['possessions', 'eFG', 'TO%',
       'OREB%', 'FTR', 'off_rating', 'def_rating', 'net_rating']


for feature in rolling_features:
       df[f"{feature}_rolling"] = df.groupby('teamName')[f"{feature}"].transform(lambda x: x.ewm(span=10).mean())

In [10]:
ewma_cols = [f"{f}_rolling" for f in rolling_features]
df[ewma_cols] = df.groupby('teamName')[ewma_cols].shift(1)

In [11]:
df = df.sort_values(by=["gameDateTimeEst"])

In [12]:
df = df.drop(columns=["points","opponentScore", "teamId", "season"])
df = df.drop(columns=rolling_features)

In [13]:
home_df = df[df["home"] == 1]
away_df = df[df["home"] == 0]

game_df = pd.merge(home_df, away_df, on='gameId', suffixes=('_home', '_away'))

In [14]:
game_df.columns

Index(['gameId', 'gameDateTimeEst_home', 'teamName_home', 'home_home',
       'win_home', 'pre_game_elo_home', 'game_date_home',
       'last_game_played_home', 'days_rest_home', 'is_B2B_home',
       'possessions_rolling_home', 'eFG_rolling_home', 'TO%_rolling_home',
       'OREB%_rolling_home', 'FTR_rolling_home', 'off_rating_rolling_home',
       'def_rating_rolling_home', 'net_rating_rolling_home',
       'gameDateTimeEst_away', 'teamName_away', 'home_away', 'win_away',
       'pre_game_elo_away', 'game_date_away', 'last_game_played_away',
       'days_rest_away', 'is_B2B_away', 'possessions_rolling_away',
       'eFG_rolling_away', 'TO%_rolling_away', 'OREB%_rolling_away',
       'FTR_rolling_away', 'off_rating_rolling_away',
       'def_rating_rolling_away', 'net_rating_rolling_away'],
      dtype='object')

In [15]:
game_df

,gameId,gameDateTimeEst_home,teamName_home,home_home,win_home,pre_game_elo_home,game_date_home,last_game_played_home,days_rest_home,is_B2B_home,...,days_rest_away,is_B2B_away,possessions_rolling_away,eFG_rolling_away,TO%_rolling_away,OREB%_rolling_away,FTR_rolling_away,off_rating_rolling_away,def_rating_rolling_away,net_rating_rolling_away
0,21200001,2012-10-30 19:00:00,Cavaliers,1,True,1346.990100,2012-10-30,NaT,NaN,False,...,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,21200002,2012-10-30 20:00:00,Heat,1,True,1660.378200,2012-10-30,NaT,NaN,False,...,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,21200003,2012-10-30 22:30:00,Lakers,1,False,1555.733900,2012-10-30,NaT,NaN,False,...,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,21200005,2012-10-31 19:00:00,76ers,1,True,1537.505900,2012-10-31,NaT,NaN,False,...,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,21200004,2012-10-31 19:00:00,Raptors,1,False,1437.342400,2012-10-31,NaT,NaN,False,...,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16742,42500304,2026-05-25 20:00:00,Cavaliers,1,False,1624.857593,2026-05-25,2026-05-23,2.0,False,...,2.0,False,93.036704,0.601428,0.128983,0.263924,0.306374,128.203809,109.875918,18.327891
16743,42500315,2026-05-26 20:30:00,Thunder,1,True,1770.420926,2026-05-26,2026-05-24,2.0,False,...,2.0,False,97.883358,0.533237,0.154104,0.288762,0.317147,117.682540,109.135156,8.547384
16744,42500316,2026-05-28 20:30:00,Spurs,1,True,1766.260845,2026-05-28,2026-05-26,2.0,False,...,2.0,False,96.658067,0.541932,0.131813,0.268761,0.289283,120.706950,113.744859,6.962091
16745,42500317,2026-05-30 20:00:00,Thunder,1,False,1763.972112,2026-05-30,2026-05-28,2.0,False,...,2.0,False,98.129045,0.526696,0.147800,0.287623,0.315700,117.780505,108.844069,8.936436


In [16]:
difference_features = ['days_rest', 'possessions_rolling', 'eFG_rolling', 'TO%_rolling',
       'OREB%_rolling', 'FTR_rolling', 'off_rating_rolling', 'def_rating_rolling', 'net_rating_rolling', "pre_game_elo"]

for features in difference_features:
    game_df[f"{features}_diff"] = game_df[f"{features}_home"] - game_df[f"{features}_away"]

In [17]:
game_df = game_df.rename(columns={"gameDateTimeEst_home":"game_date"})
final_df = game_df[['game_date','teamName_home','teamName_away', 'pre_game_elo_home', 'is_B2B_home',
       'pre_game_elo_away', 'is_B2B_away', 'pre_game_elo_diff','days_rest_diff','possessions_rolling_diff', 'eFG_rolling_diff',
       'TO%_rolling_diff', 'OREB%_rolling_diff', 'FTR_rolling_diff',
       'off_rating_rolling_diff', 'def_rating_rolling_diff',
       'net_rating_rolling_diff', 'win_home']]

In [18]:
final_df.columns

Index(['game_date', 'teamName_home', 'teamName_away', 'pre_game_elo_home',
       'is_B2B_home', 'pre_game_elo_away', 'is_B2B_away', 'pre_game_elo_diff',
       'days_rest_diff', 'possessions_rolling_diff', 'eFG_rolling_diff',
       'TO%_rolling_diff', 'OREB%_rolling_diff', 'FTR_rolling_diff',
       'off_rating_rolling_diff', 'def_rating_rolling_diff',
       'net_rating_rolling_diff', 'win_home'],
      dtype='object')

In [19]:
final_df = final_df.dropna()

In [20]:
final_df = final_df.reset_index(drop=True)

In [21]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16727 entries, 0 to 16726
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   game_date                 16727 non-null  datetime64[ns]
 1   teamName_home             16727 non-null  object        
 2   teamName_away             16727 non-null  object        
 3   pre_game_elo_home         16727 non-null  float64       
 4   is_B2B_home               16727 non-null  bool          
 5   pre_game_elo_away         16727 non-null  float64       
 6   is_B2B_away               16727 non-null  bool          
 7   pre_game_elo_diff         16727 non-null  float64       
 8   days_rest_diff            16727 non-null  float64       
 9   possessions_rolling_diff  16727 non-null  float64       
 10  eFG_rolling_diff          16727 non-null  float64       
 11  TO%_rolling_diff          16727 non-null  float64       
 12  OREB%_rolling_diff

In [22]:
final_df.shape

(16727, 18)

In [23]:
final_df.to_csv("/Users/abhaybapat/Desktop/Data Science Project/betting_classification_model/data/final_dataset.csv")